# SpaceX Data Wrangling

**Purpose:** inspect data quality, prepare the binary landing label, and create model-ready features.

In [1]:
from pathlib import Path
def resolve_data(filename):
    candidates = [Path("data")/filename, Path("../data")/filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(filename)

import pandas as pd
df=pd.read_csv(resolve_data("spacex_launch_data.csv"))
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
print(df.isna().sum())
df.head()


Shape: (20, 6)
Duplicate rows: 0

Missing values:
FlightNumber    0
LaunchSite      0
Orbit           0
PayloadMass     0
Class           0
Year            0
dtype: int64


,FlightNumber,LaunchSite,Orbit,PayloadMass,Class,Year
0,1,CCAFS SLC 40,LEO,6123,0,2010
1,2,CCAFS SLC 40,GTO,4535,1,2012
2,3,KSC LC 39A,ISS,5300,1,2014
3,4,VAFB SLC 4E,SSO,4750,1,2015
4,5,CCAFS SLC 40,LEO,6200,1,2016


In [2]:

df["Class"] = df["Class"].astype(int)
features = pd.get_dummies(
    df[["FlightNumber","LaunchSite","Orbit","PayloadMass","Year"]],
    columns=["LaunchSite","Orbit"],
    dtype=float
)
target = df["Class"]
print("Feature matrix shape:", features.shape)
print("Target distribution:")
print(target.value_counts())
features.head()


Feature matrix shape: (20, 11)
Target distribution:
Class
1    16
0     4
Name: count, dtype: int64


,FlightNumber,PayloadMass,Year,LaunchSite_CCAFS SLC 40,LaunchSite_KSC LC 39A,LaunchSite_VAFB SLC 4E,Orbit_GTO,Orbit_ISS,Orbit_LEO,Orbit_PO,Orbit_SSO
0,1,6123,2010,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,2,4535,2012,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,5300,2014,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,4,4750,2015,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,5,6200,2016,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Preparation decisions

- Keep Falcon 9 analytical records.
- Review missing values and duplicates.
- Use `Class` as the binary landing target: **1 = success, 0 = unsuccessful**.
- One-hot encode categorical launch-site and orbit variables.
- Preserve numerical flight, payload and year fields for modeling.